# Stock Price Prediction Using Artificial Neural Network
Academic Mini Project — End-to-End Pipeline

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from preprocessing  import load_dataset, inspect_data, preprocess
from train_ann      import build_ann, train_ann, save_model
from train_baseline import train_linear_regression, train_random_forest
from evaluation     import (compute_metrics, build_metrics_table, save_metrics_csv,
                             plot_closing_price, plot_volume, plot_training_loss,
                             plot_actual_vs_predicted, plot_model_comparison, plot_residuals)

## 1. Load Dataset

In [ ]:
DATA_PATH = '../data/stock_data.csv'
df = load_dataset(DATA_PATH)
print(df.shape)
df.head()

## 2. Inspect Data

In [ ]:
info = inspect_data(df)
for k, v in info.items():
    print(f'{k}: {v}')

## 3. Visualize Historical Data

In [ ]:
fig1 = plot_closing_price(df)
plt.show()
fig2 = plot_volume(df)
plt.show()

## 4. Preprocess Data

In [ ]:
FEATURES = ['Open', 'High', 'Low', 'Volume']
X_train, X_test, y_train, y_test, scaler_X, scaler_y, df_clean = preprocess(df, FEATURES)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5. Train Baseline Models

In [ ]:
lr_model = train_linear_regression(X_train, y_train)
rf_model = train_random_forest(X_train, y_train)

y_test_inv = scaler_y.inverse_transform(y_test)
lr_pred = scaler_y.inverse_transform(lr_model.predict(X_test).reshape(-1,1))
rf_pred = scaler_y.inverse_transform(rf_model.predict(X_test).reshape(-1,1))

## 6. Train ANN

In [ ]:
ann_model, history = train_ann(X_train, y_train, X_test, y_test, epochs=150, batch_size=32)
save_model(ann_model, '../models/ann_model.keras')
ann_pred = scaler_y.inverse_transform(ann_model.predict(X_test, verbose=0))

## 7. Visualize Results

In [ ]:
plot_training_loss(history); plt.show()
plot_actual_vs_predicted(y_test_inv, ann_pred, 'ANN'); plt.show()
plot_residuals(y_test_inv, ann_pred, 'ANN'); plt.show()

## 8. Evaluation Metrics

In [ ]:
results = {
    'Linear Regression': compute_metrics(y_test_inv, lr_pred),
    'Random Forest':     compute_metrics(y_test_inv, rf_pred),
    'ANN':               compute_metrics(y_test_inv, ann_pred),
}
metrics_df = build_metrics_table(results)
save_metrics_csv(metrics_df, '../outputs/metrics.csv')
print(metrics_df.to_string(index=False))

## 9. Model Comparison Chart

In [ ]:
plot_model_comparison(metrics_df)
plt.show()